## Step 0: Mounting Google Drive and Importing Libraries

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/multimodal-xray-agent
!ls

In [ ]:
!pip install llmcompressor

In [2]:
import gc
import json
import torch
import numpy
import transformers

from pathlib import Path
from datasets import Dataset
from huggingface_hub import login
from llmcompressor import oneshot
from llmcompressor.modifiers.awq import AWQModifier
from transformers import AutoTokenizer, AutoModelForCausalLM
from llmcompressor.modifiers.smoothquant import SmoothQuantModifier

In [ ]:
login()

In [5]:
print(transformers.__version__)
print(torch.__version__)
print(numpy.__version__)

4.52.4
2.6.0+cu124
1.26.4


## Step 1: Verifying GPU and Environment

In [6]:
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    device = torch.device("cuda")
    print(f"GPU detected: {device_name}")
else:
    device = torch.device("cpu")
    print("GPU not detected. Falling back to CPU.")

print(f"Running on device: {device}")

GPU detected: NVIDIA A100-SXM4-40GB
Running on device: cuda


## Step 2: Prepare Domain-Specific Calibration Dataset

In [7]:
PROJECT_ROOT = Path("/content/drive/MyDrive/multimodal-xray-agent")
QA_PAIRS_PATH = PROJECT_ROOT / "data" / "qapairs" / "top_700_qa_pairs.jsonl"
SAVE_PATH = PROJECT_ROOT / "models" / "llama-awq"

SAVE_PATH.mkdir(parents=True, exist_ok=True)

In [8]:
# Load the JSONL file
with open(QA_PAIRS_PATH, "r") as f:
  data = [json.loads(line) for line in f]
  print(f"Successfully loaded {len(data)} records.")

Successfully loaded 700 records.


In [9]:
# We use the 'answer' texts (radiology impressions) for calibration
calibration_text = [item['answer'] for item in data]

In [10]:
# The AWQ authors recommend a small, representative subset. 256 samples is a standard choice
num_calibration_samples = 128
calibration_dataset = calibration_text[:num_calibration_samples]

In [11]:
print(calibration_dataset[0])

1. Severe emphysema. 2. Irregular, pleural-parenchymal opacity in left upper lobe. This may irregular pleural-parenchymal scarring, however, recommend comparison with more remote outside imaging, if available to determine long-term stability. If none are available, recommend short-term [REDACTED] in 3 to 4 months. Evaluation of coronal and sagittal reformatted images from the outside study would also be helpful. These were not [REDACTED] available at the outside institution. Malignancy cannot be confidently excluded on the available images


## Step 3: Load the Base Llama Model

In [12]:
MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True # It's good practice for community models
)

In [15]:
print(f"- Memory Footprint: {model.get_memory_footprint() / 1e9:.2f} GB")

- Memory Footprint: 6.43 GB


## Step 4: Quantize the Model

In [16]:
# This tells the system to apply 4-bit asymmetric AWQ to the linear layers of the model
recipe = [
    SmoothQuantModifier(smoothing_strength=0.8),
    AWQModifier(ignore=["lm_head"], scheme="W4A16_ASYM", targets=["Linear"]),
]

In [17]:
# The 'oneshot' function expects a Hugging Face Dataset object.
# We will convert our Python list of calibration strings into one.
calibration_hf_dataset = Dataset.from_dict({"text": calibration_dataset})

In [ ]:
# This is the main function call. It orchestrates the entire AWQ process
# It will use our calibration data to produce the highest-quality result
oneshot(
    model=model,
    dataset=calibration_hf_dataset,
    recipe=recipe,
    max_seq_length=512,
    num_calibration_samples=num_calibration_samples,
    text_column="text"
)